In [11]:
import torch
import torch.nn as nn
import pickle
import numpy as np
import os
import sys

# Nếu cần, thêm đường dẫn chứa DeepFM class
sys.path.append("models")
from deepfm import DeepFM

In [12]:
# Đường dẫn đến model và file encode
model_path = "../Model/Deep FM/article/deepfm_article.pth"
feature_index_path = "../Model/Deep FM/article/feature_index_dict.pkl"
scaler_path = "../Model/Deep FM/article/scaler.pkl"

with open(feature_index_path, "rb") as f:
    feature_index = pickle.load(f)

with open(scaler_path, "rb") as f:
    scaler = pickle.load(f)

categorical_cols = ["user_id", "item_id", "category_second", "category_first", "gender", "age"]
numerical_cols = ["exposure_count", "click_count", "like_count", "comment_count",
                  "read_percentage", "item_score1", "item_score2", "item_score3", "read_time"]

cat_dims = [len(feature_index[col]) for col in categorical_cols]
num_dim = len(numerical_cols)

# Load mô hình
model = DeepFM(cat_dims=cat_dims, num_dim=num_dim)
model.load_state_dict(torch.load(model_path, map_location=torch.device("cpu")))
model.eval()

print("✅ Model loaded.")


✅ Model loaded.


In [13]:
# Lấy 5 item_id khác nhau từ feature_index
test_item_ids = list(feature_index["item_id"].keys())[:5]
print("🧪 Testing item_ids:", test_item_ids)

# Các input categorical cố định
user_idx = feature_index["user_id"].get("unknown_user", 0)
cat1_idx = 0
cat2_idx = 0
gender_idx = feature_index["gender"].get("1", 0)
age_idx = feature_index["age"].get("3", 0)

# Numeric features: dùng giá trị random để tạo biến động
numeric_inputs = np.random.uniform(0.5, 1.5, size=(5, num_dim))
numeric_scaled = scaler.transform(numeric_inputs)
num_tensor = torch.tensor(numeric_scaled, dtype=torch.float32)


🧪 Testing item_ids: [438, 548, 602, 774, 1093]


c:\Python311\Lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(


In [14]:
# Tạo tensor categorical cho từng item_id
cat_tensor = []
for item_id in test_item_ids:
    item_idx = feature_index["item_id"].get(item_id, 0)
    cat_row = [user_idx, item_idx, cat2_idx, cat1_idx, gender_idx, age_idx]
    cat_tensor.append(cat_row)

cat_tensor = torch.tensor(cat_tensor, dtype=torch.long)

# Dự đoán
with torch.no_grad():
    scores = model(cat_tensor, num_tensor).squeeze().numpy()

# In kết quả
for item_id, score in zip(test_item_ids, scores):
    print(f"Item {item_id} → Score: {score:.4f}")


Item 438 → Score: 1.0000
Item 548 → Score: 1.0000
Item 602 → Score: 1.0000
Item 774 → Score: 1.0000
Item 1093 → Score: 1.0000
